# HMDA Mortgage Intelligence — historical exploration

A compact walkthrough of the existing Parquet samples, baseline and preferred model,
2025 evaluation, feature importance, and descriptive demographic monitoring.

**Default mode loads trusted saved models; it does not fit models or write artifacts.**
Optional fitting is in memory only and requires `RUN_TRAINING = True` below.
For reproducible training and durable, isolated artifacts, use
`python workflows/train_hmda.py` instead; see [the handoff](docs/HANDOFF.md).
Use the prepared project Python environment; this notebook does not install packages.
Historical source downloads are restored below but require a separate explicit opt-in.
Run from the repository root.

**Target:** actions 1 (originated) and 2 (approved but not accepted) are positive;
action 3 is denied. Origination is reported separately.

**Historical limitation:** earlier versions of this notebook explored multiple model
variants and 2025 thresholds. Cleaning the notebook does not undo that exposure.
2025 is excluded from fitting here, but is not a previously untouched final test set.
The earlier exploration remains in Git history.


In [ ]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.inspection import permutation_importance
from threadpoolctl import threadpool_limits

from backend.config import DATA_DIR, FEATURES, NUMERIC, CATEGORICAL, YEARS, TARGET_NOTE
from backend.data_service import prepare_features
from workflows.hmda_input import validate
from workflows.train_hmda import split_frames, metrics, select_threshold, fairness

RUN_TRAINING = False  # Explicit opt-in: refit copies of the saved pipelines in memory.
MODEL_DIR = DATA_DIR / 'models'
print(TARGET_NOTE)
print('Mode:', 'in-memory exploratory refit' if RUN_TRAINING else 'saved-model evaluation')


## Historical source ingestion: CFPB CSV → annual Parquets

The original notebook created the mounted samples; they were not downloaded as Parquet.
The source-preparation code below is restored from the original notebook in Git:
2021 was processed separately, followed by a 2022–2025 loop.

Each path downloads the public snapshot ZIP, reads its CSV in chunks, filters purchase /
first-lien / principal-residence / site-built / 1–4-unit applications with actions 1–3,
keeps up to 25,000 records using seed 42, maps actions 1+2 to positive and 3 to denied,
and writes an annual Parquet. The later-year loop also records a source manifest and
removes its temporary ZIPs.

**This is historical preparation code, not evidence of a fresh download.** Original
source bytes are no longer available here, and archive access previously returned HTTP
403. The source contract describes the provenance limits.

Safety changes to the restored code: it is wrapped in functions, disabled by default,
writes only to a fresh isolated run directory, and uses exclusive Parquet/manifest writes.
Duplicate download attempts and package-install cells are omitted. The original sampling
logic is retained; the maintained `workflows/hmda_input.py` implementation adds stronger
validation and download limits and is preferred for operational ingestion.

Leave `RUN_SOURCE_INGESTION = False` for this exercise: the following analysis already
uses the five mounted Parquets. Enabling it attempts five potentially large external
downloads and requires network access, disk capacity, and a writable artifact root.
It does not switch the analysis to these newly generated samples or modify the App Dataset.


In [ ]:
def historical_ingestion_2021(output_dir, temp_dir):
    if (output_dir / "hmda_2021.parquet").exists():
        raise FileExistsError("Refusing to overwrite the 2021 Parquet")

    from pathlib import Path
    import requests
    import zipfile
    import pandas as pd
    import hashlib

    YEAR = 2021

    HMDA_URL = "https://files.ffiec.cfpb.gov/static-data/snapshot/2021/2021_public_lar_csv.zip"

    DATASET_DIR = output_dir
    TEMP_DIR = temp_dir
    TEMP_DIR.mkdir(parents=True, exist_ok=True)

    ZIP_PATH = TEMP_DIR / f"hmda_{YEAR}.zip"
    OUTPUT_PATH = DATASET_DIR / f"hmda_{YEAR}.parquet"

    print("Dataset directory:", DATASET_DIR)
    print("Output:", OUTPUT_PATH)

    import requests
    import hashlib
    import zipfile

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*",
    }

    with requests.get(
        HMDA_URL,
        headers=headers,
        stream=True,
        timeout=(30, 120)
    ) as response:

        print("HTTP Status:", response.status_code)
        print("Final URL:", response.url)
        response.raise_for_status()

        sha256 = hashlib.sha256()
        total_bytes = 0

        with open(ZIP_PATH, "xb") as file:
            for block in response.iter_content(1024 * 1024):
                if block:
                    file.write(block)
                    sha256.update(block)
                    total_bytes += len(block)

    print("Downloaded MB:", round(total_bytes / 1e6, 1))
    print("SHA256:", sha256.hexdigest())
    print("ZIP valid:", zipfile.is_zipfile(ZIP_PATH))

    import zipfile
    import pandas as pd

    with zipfile.ZipFile(ZIP_PATH) as archive:
        csv_files = [
            name for name in archive.namelist()
            if name.lower().endswith(".csv")
        ]

        print("CSV files:", csv_files)

        with archive.open(csv_files[0]) as file:
            preview = pd.read_csv(file, nrows=5, dtype=str)

    print("Number of columns:", len(preview.columns))
    print("Columns:", preview.columns.tolist())

    display(preview.head())

    import pandas as pd
    import numpy as np
    import zipfile
    from pathlib import Path

    YEAR = 2021
    SAMPLE_SIZE = 25_000
    CHUNK_SIZE = 100_000

    OUTPUT_PATH = output_dir / f"hmda_{YEAR}.parquet"

    rng = np.random.default_rng(42)

    required = [
        "action_taken",
        "loan_purpose",
        "lien_status",
        "occupancy_type",
        "construction_method",
        "total_units",
    ]

    # Columns to retain for modeling and fairness analysis
    keep_columns = [
        "activity_year",
        "lei",
        "state_code",
        "county_code",
        "census_tract",
        "action_taken",
        "loan_type",
        "loan_purpose",
        "lien_status",
        "occupancy_type",
        "construction_method",
        "total_units",
        "loan_amount",
        "income",
        "debt_to_income_ratio",
        "combined_loan_to_value_ratio",
        "loan_term",
        "property_value",
        "applicant_age",
        "applicant_sex",
        "applicant_race_1",
        "applicant_ethnicity_1",
        "derived_race",
        "derived_ethnicity",
        "derived_sex",
        "denial_reason_1",
    ]

    available_columns = [
        col for col in keep_columns if col in preview.columns
    ]

    missing = set(required) - set(preview.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    sample = None
    processed_total = 0
    eligible_total = 0

    with zipfile.ZipFile(ZIP_PATH) as archive:
        with archive.open(csv_files[0]) as file:

            for chunk in pd.read_csv(
                file,
                chunksize=CHUNK_SIZE,
                dtype=str,
                usecols=available_columns,
            ):
                processed_total += len(chunk)

                # Filter to comparable mortgage applications
                filtered = chunk[
                    chunk["action_taken"].isin(["1", "2", "3"])
                    & chunk["loan_purpose"].eq("1")
                    & chunk["lien_status"].eq("1")
                    & chunk["occupancy_type"].eq("1")
                    & chunk["construction_method"].eq("1")
                    & chunk["total_units"].isin(["1", "2", "3", "4"])
                ].copy()

                n = len(filtered)
                eligible_total += n

                if n == 0:
                    continue

                # Fixed-seed random sample across the full dataset
                filtered["_priority"] = rng.random(n)

                if sample is None:
                    sample = filtered
                else:
                    sample = pd.concat(
                        [sample, filtered],
                        ignore_index=True
                    )

                if len(sample) > SAMPLE_SIZE:
                    sample = sample.nsmallest(
                        SAMPLE_SIZE,
                        "_priority"
                    ).copy()

                if processed_total % 1_000_000 == 0:
                    print(
                        f"Processed {processed_total:,} rows; "
                        f"eligible {eligible_total:,}"
                    )

    if sample is None or sample.empty:
        raise ValueError("No eligible mortgage applications found")

    sample = sample.drop(columns=["_priority"])
    sample["source_year"] = YEAR

    # 1 = originated, 2 = approved but not accepted
    # 3 = denied
    sample["approved"] = sample["action_taken"].isin(
        ["1", "2"]
    ).astype(int)

    with OUTPUT_PATH.open("xb") as parquet_file:
        sample.to_parquet(parquet_file, index=False, compression="snappy")

    print("\nSUCCESS!")
    print("Total records scanned:", f"{processed_total:,}")
    print("Eligible applications:", f"{eligible_total:,}")
    print("Sample size:", len(sample))
    print("Approval rate:", round(sample["approved"].mean() * 100, 2), "%")
    print("Parquet saved:", OUTPUT_PATH)
    print("File size:", round(OUTPUT_PATH.stat().st_size / 1e6, 2), "MB")


In [ ]:
def historical_ingestion_2022_2025(output_dir, temp_dir):
    protected_names = [f"hmda_{year}.parquet" for year in range(2022, 2026)] + ["ingestion_manifest_2022_2025.json"]
    if any((output_dir / name).exists() for name in protected_names):
        raise FileExistsError("Refusing to overwrite annual Parquets or the source manifest")

    import requests
    import pandas as pd
    import numpy as np
    import zipfile
    import hashlib
    import json
    import gc
    from pathlib import Path

    # Configuration
    YEARS = [2022, 2023, 2024, 2025]
    SAMPLE_SIZE = 25_000
    CHUNK_SIZE = 100_000
    SEED = 42

    DATASET_DIR = output_dir
    TEMP_DIR = temp_dir

    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    TEMP_DIR.mkdir(parents=True, exist_ok=True)

    # Verify these URLs against the official HMDA Snapshot pages.
    # The 2021 URL worked, but the other URLs are not yet verified.
    URLS = {
        year: (
            f"https://files.ffiec.cfpb.gov/static-data/"
            f"snapshot/{year}/{year}_public_lar_csv.zip"
        )
        for year in YEARS
    }

    KEEP_COLUMNS = [
        "activity_year",
        "lei",
        "state_code",
        "county_code",
        "census_tract",
        "action_taken",
        "loan_type",
        "loan_purpose",
        "lien_status",
        "occupancy_type",
        "construction_method",
        "total_units",
        "loan_amount",
        "income",
        "debt_to_income_ratio",
        "combined_loan_to_value_ratio",
        "loan_term",
        "property_value",
        "applicant_age",
        "applicant_sex",
        "applicant_race_1",
        "applicant_ethnicity_1",
        "derived_race",
        "derived_ethnicity",
        "derived_sex",
        "denial_reason_1",
    ]

    REQUIRED = [
        "action_taken",
        "loan_purpose",
        "lien_status",
        "occupancy_type",
        "construction_method",
        "total_units",
    ]

    HEADERS = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "*/*",
    }

    manifest = {}

    for year in YEARS:
        print(f"\n{'='*50}")
        print(f"PROCESSING HMDA {year}")
        print(f"{'='*50}")

        url = URLS[year]
        zip_path = TEMP_DIR / f"hmda_{year}.zip"
        output_path = DATASET_DIR / f"hmda_{year}.parquet"

        if output_path.exists():
            print(f"Already exists: {output_path}")
            print("Skipping this year.")
            continue

        # 1. Download ZIP
        print("Downloading:", url)

        sha256 = hashlib.sha256()
        total_bytes = 0

        with requests.get(
            url,
            headers=HEADERS,
            stream=True,
            timeout=(30, 120)
        ) as response:

            response.raise_for_status()

            with open(zip_path, "xb") as f:
                for block in response.iter_content(1024 * 1024):
                    if block:
                        f.write(block)
                        sha256.update(block)
                        total_bytes += len(block)

        if not zipfile.is_zipfile(zip_path):
            raise ValueError(f"Invalid ZIP for {year}")

        print(f"Downloaded: {total_bytes / 1e6:.1f} MB")

        # 2. Process CSV in chunks
        rng = np.random.default_rng(SEED)
        sample = None
        processed_total = 0
        eligible_total = 0

        with zipfile.ZipFile(zip_path) as archive:
            csv_files = [
                name for name in archive.namelist()
                if name.lower().endswith(".csv")
            ]

            if len(csv_files) != 1:
                raise ValueError(
                    f"Expected one CSV for {year}, found {csv_files}"
                )

            with archive.open(csv_files[0]) as file:
                for chunk in pd.read_csv(
                    file,
                    chunksize=CHUNK_SIZE,
                    dtype=str,
                    usecols=lambda c: c in KEEP_COLUMNS,
                ):
                    processed_total += len(chunk)

                    missing = set(REQUIRED) - set(chunk.columns)
                    if missing:
                        raise ValueError(
                            f"{year}: Missing columns {missing}"
                        )

                    filtered = chunk[
                        chunk["action_taken"].isin(["1", "2", "3"])
                        & chunk["loan_purpose"].eq("1")
                        & chunk["lien_status"].eq("1")
                        & chunk["occupancy_type"].eq("1")
                        & chunk["construction_method"].eq("1")
                        & chunk["total_units"].isin(["1", "2", "3", "4"])
                    ].copy()

                    n = len(filtered)
                    eligible_total += n

                    if n == 0:
                        continue

                    filtered["_priority"] = rng.random(n)

                    if sample is None:
                        sample = filtered
                    else:
                        sample = pd.concat(
                            [sample, filtered],
                            ignore_index=True
                        )

                    if len(sample) > SAMPLE_SIZE:
                        sample = sample.nsmallest(
                            SAMPLE_SIZE, "_priority"
                        ).copy()

                    if processed_total % 1_000_000 == 0:
                        print(
                            f"Scanned {processed_total:,} rows; "
                            f"eligible {eligible_total:,}"
                        )

        if sample is None or sample.empty:
            raise ValueError(f"No eligible records for {year}")

        # 3. Prepare final data
        sample = sample.drop(columns=["_priority"])
        sample["source_year"] = year
        sample["approved"] = (
            sample["action_taken"].isin(["1", "2"])
        ).astype(int)

        # 4. Save Parquet
        with output_path.open("xb") as parquet_file:
            sample.to_parquet(parquet_file, index=False, compression="snappy")

        approval_rate = sample["approved"].mean()

        manifest[str(year)] = {
            "source_url": url,
            "source_sha256": sha256.hexdigest(),
            "rows_scanned": processed_total,
            "eligible_rows": eligible_total,
            "sample_rows": len(sample),
            "sample_seed": SEED,
            "approval_rate": float(approval_rate),
            "output_file": output_path.name,
        }

        print(f"\nSUCCESS: {year}")
        print("Rows scanned:", f"{processed_total:,}")
        print("Eligible:", f"{eligible_total:,}")
        print("Saved rows:", len(sample))
        print("Approval rate:", f"{approval_rate:.2%}")
        print("Output:", output_path)

        # Remove temporary ZIP to conserve disk space
        zip_path.unlink()
        del sample
        gc.collect()

    # 5. Save metadata
    manifest_path = DATASET_DIR / "ingestion_manifest_2022_2025.json"

    with open(manifest_path, "x") as f:
        json.dump(manifest, f, indent=2)

    print("\nALL AVAILABLE YEARS PROCESSED")
    print("Manifest:", manifest_path)


In [ ]:
RUN_SOURCE_INGESTION = False

if RUN_SOURCE_INGESTION:
    import tempfile
    from pathlib import Path

    ingestion_root = Path('/mnt/artifacts/hmda-historical-ingestion')
    ingestion_root.mkdir(parents=True, exist_ok=True)
    ingestion_run = Path(tempfile.mkdtemp(prefix='run-', dir=ingestion_root))
    ingestion_data = ingestion_run / 'data'
    ingestion_data.mkdir()
    ingestion_downloads = ingestion_run / 'downloads'
    ingestion_downloads.mkdir()
    historical_ingestion_2021(ingestion_data, ingestion_downloads)
    historical_ingestion_2022_2025(ingestion_data, ingestion_downloads)
    print('New historical-ingestion outputs:', ingestion_run)
else:
    print('Source downloads skipped. Analysis uses the existing mounted Parquets.')


## 1. Parquet ingestion and source provenance

There are two stages to distinguish in the presentation:

```text
Historical data preparation (original notebook, retained in Git history)
CFPB public HMDA snapshot ZIP/CSV
  → cohort filtering → fixed-seed annual sampling → target mapping
  → hmda_2021.parquet … hmda_2025.parquet in the Domino Dataset

Current offline ingestion (executed by the next cell)
Existing mounted Parquets → pandas.read_parquet → schema/year/cohort/label validation
  → in-memory annual frames → temporal split
```

The next cell **does ingest the Parquets**: it reads each file and validates it before
using its records. It does not repeat the historical download or overwrite those files.
For this exercise, the existing Parquets are the accepted input.

The cohort is home purchase, first lien, principal residence, site-built, 1–4 units,
and actions 1–3. Actions 1 and 2 map to positive; action 3 maps to denied.
The existing samples have 25,000 rows per year; the cell displays actual counts rather
than assuming them. Equal annual samples are not national volumes.

**Reproducible ingestion implementation:** [workflows/hmda_input.py](workflows/hmda_input.py)
contains `validate`, `sample_archive`, and `ingest`. Archive sampling reads chunks across
the entire eligible stream with seed 42; mounted mode preserves the existing sample.
The workflow records input/output hashes and writes isolated validated copies.

To run only ingestion and validation as a separate Job, without model training:

```bash
python workflows/train_hmda.py --source mounted --ingest-only
```

**Optional archive mode — not available with the current files.** No ZIP archives were
found in `/mnt/data/HMDA_Project`, and `/tmp/hmda_ingestion` does not exist
(checked 2026-11-05). The current exercise uses the five existing Parquets.
`/path/to/archives` below is a placeholder, not an existing project directory.
Only if you separately obtain the official annual ZIP archives could you run:

```bash
python workflows/train_hmda.py --source archives --input-dir /path/to/archives --ingest-only
```

That optional, user-supplied archive directory would need `2021_public_lar_csv.zip` through
`2025_public_lar_csv.zip`. Both commands write to a unique directory under
`/mnt/artifacts/hmda-training/`; neither overwrites the existing Dataset. These are
instructions, not commands automatically executed by this notebook.

**Provenance limits:** `ingestion_manifest_2022_2025.json`, when present, is historical
source evidence, not a freshly verified archive-to-sample chain. The 2021 provenance
comes from the original notebook. Live archive access was previously blocked with HTTP
403; no fresh download is claimed. See [the source contract](docs/HMDA_SOURCE.md).


In [ ]:
# Offline ingestion: read and validate every required annual Parquet.
frames = {
    year: validate(pd.read_parquet(DATA_DIR / f'hmda_{year}.parquet'), year, already_filtered=True)
    for year in YEARS
}
summary = []
for year, frame in frames.items():
    action = pd.to_numeric(frame['action_taken'])
    summary.append({
        'year': year,
        'input_file': str(DATA_DIR / f'hmda_{year}.parquet'),
        'columns': len(frame.columns),
        'records': len(frame),
        'originated': int(action.eq(1).sum()),
        'approved_not_accepted': int(action.eq(2).sum()),
        'denied': int(action.eq(3).sum()),
        'positive_rate': float(frame['approved'].mean()),
    })
display(pd.DataFrame(summary))

# Show the recorded source evidence if available; never invent missing provenance.
manifest_path = DATA_DIR / 'ingestion_manifest_2022_2025.json'
if manifest_path.is_file():
    upstream_manifest = json.loads(manifest_path.read_text())
    display(upstream_manifest)
else:
    print('Historical ingestion manifest unavailable; see the source contract and Git history.')


## 2. Temporal split and six predictors

- **2021–2023:** training (75,000 records in the existing samples).
- **2024:** validation (25,000 records); threshold selection and permutation importance.
- **2025:** holdout evaluation (25,000 records); never passed to `.fit()`.

Numeric normalization and categorical `Missing` labels use the shared helper.
Learned medians, scaling where applicable, and encodings belong to the model pipeline.
Demographics and outcome-related fields are excluded from predictors.


In [ ]:
train, val, test = split_frames(frames)
X_train, X_val, X_test = [prepare_features(frame) for frame in (train, val, test)]
y_train, y_val, y_test = [frame['approved'].astype(int) for frame in (train, val, test)]
display(pd.DataFrame([
    {'partition': name, 'records': len(frame), 'years': sorted(frame['source_year'].unique().tolist())}
    for name, frame in [('train', train), ('validation', val), ('holdout', test)]
]))
display(pd.DataFrame({'feature': FEATURES, 'type': ['numeric'] * len(NUMERIC) + ['categorical'] * len(CATEGORICAL)}))


## 3. Existing baseline and preferred pipelines

The baseline uses median imputation, numeric scaling, one-hot encoding, and class-balanced
logistic regression. The preferred model uses median imputation, dense one-hot encoding,
and histogram gradient boosting. Inspect the saved pipelines below for exact parameters.

Only trusted project artifacts are deserialized. The optional refit clones their configurations
and fits on 2021–2023 only; it never replaces saved files. This historical configuration is
not the same as the standalone workflow, which explicitly disables early stopping.


In [ ]:
specs = {
    'Logistic Regression - Baseline': ('mortgage_approval_model.joblib', 'model_metrics.json'),
    'Gradient Boosting - Preferred': ('mortgage_gradient_boosting.joblib', 'gradient_boosting_metrics.json'),
}
models, metadata = {}, {}
for name, (model_file, metrics_file) in specs.items():
    metadata[name] = json.loads((MODEL_DIR / metrics_file).read_text())
    if metadata[name]['features'] != FEATURES or metadata[name]['test_year'] != 2025:
        raise ValueError(f'{name}: saved feature order or test year differs from this walkthrough')
    saved_model = joblib.load(MODEL_DIR / model_file)
    if list(saved_model.classes_) != [0, 1]:
        raise ValueError(f'{name}: expected class order [0, 1]')
    if RUN_TRAINING:
        models[name] = clone(saved_model)
        with threadpool_limits(limits=2):
            models[name].fit(X_train, y_train)
    else:
        models[name] = saved_model
    print(name)
    display(models[name])


## 4. Validation and threshold policy

Saved-model mode preserves each saved threshold (baseline 0.5; preferred 0.89).
For an explicitly requested in-memory refit, keep the baseline at 0.5 and select the
boosting threshold using **2024 balanced accuracy only**, on 0.50–0.99 in steps of 0.01,
with ties resolved to the lowest threshold. Do not tune using the 2025 results below.


In [ ]:
thresholds, validation_rows = {}, []
for name, model in models.items():
    with threadpool_limits(limits=2):
        scores = model.predict_proba(X_val)[:, 1]
    threshold = metadata[name]['threshold']
    if RUN_TRAINING and name == 'Gradient Boosting - Preferred':
        threshold, threshold_candidates = select_threshold(y_val, scores)
        display(pd.DataFrame(threshold_candidates).sort_values(
            ['balanced_accuracy', 'threshold'], ascending=[False, True]
        ).head(10))
    thresholds[name] = threshold
    result = metrics(y_val, scores, threshold)
    validation_rows.append({'model': name, 'threshold': threshold, **{
        key: result[key] for key in ['roc_auc', 'balanced_accuracy', 'denial_recall', 'accuracy']
    }})
display(pd.DataFrame(validation_rows))


## 5. Fixed 2025 evaluation

Both models use the same holdout and their fixed thresholds. Saved-model mode verifies
all four metrics against the existing reports with absolute tolerance `1e-10` and zero
relative tolerance. Refit results are labeled exploratory rather than compared as if they
were the saved artifacts. Confusion matrices use rows/columns `[denied, positive]`.

An always-positive classifier has high raw accuracy in this imbalanced cohort but zero
denial recall. Balanced accuracy and ROC-AUC provide additional context.


In [ ]:
holdout_scores, holdout_results, comparison_rows = {}, {}, []
for name, model in models.items():
    with threadpool_limits(limits=2):
        scores = model.predict_proba(X_test)[:, 1]
    result = metrics(y_test, scores, thresholds[name])
    if not RUN_TRAINING:
        for key, expected in metadata[name]['test_metrics'].items():
            if not np.isclose(result[key], expected, atol=1e-10, rtol=0):
                raise ValueError(f'{name}: {key} differs from the saved evaluation')
    holdout_scores[name], holdout_results[name] = scores, result
    comparison_rows.append({'model': name, 'threshold': thresholds[name], **{
        key: result[key] for key in ['roc_auc', 'balanced_accuracy', 'denial_recall', 'accuracy']
    }})
print('Evaluation:', 'exploratory refits' if RUN_TRAINING else 'verified saved models')
print(f'Always-positive accuracy: {y_test.mean():.3%}')
display(pd.DataFrame(comparison_rows))
for name, result in holdout_results.items():
    print(name)
    display(pd.DataFrame(result['confusion_matrix'],
                         index=['Actual denied', 'Actual positive'],
                         columns=['Predicted denied', 'Predicted positive']))


## 6. Preferred-model permutation importance

Compute ROC-AUC decrease on **2024 validation**, using five permutations and seed 42.
The model is fixed during permutation. Importance measures predictive contribution,
not causal effects. Results stay in memory; the dashboard's saved CSV is not overwritten.


In [ ]:
preferred_name = 'Gradient Boosting - Preferred'
with threadpool_limits(limits=2):
    importance = permutation_importance(
        models[preferred_name], X_val, y_val,
        scoring='roc_auc', n_repeats=5, random_state=42, n_jobs=1,
    )
importance_table = pd.DataFrame({
    'feature': FEATURES,
    'importance_mean': importance.importances_mean,
    'importance_std': importance.importances_std,
}).sort_values('importance_mean', ascending=False)
display(importance_table)


## 7. Descriptive demographic monitoring

Use the shared workflow report for race, ethnicity, and sex. Distinguish historical
origination (action 1), historical positives (actions 1 + 2), and model classifications.
Rates with zero denominators are undefined, not zero. Groups are flagged if they have
fewer than 100 records, 30 denials, or 30 positives. Unavailable demographics remain visible.

Supported group-rate ranges require at least two named, sufficiently supported groups.
These are unadjusted descriptive results without confidence intervals or causal controls;
they are not legal findings or individual underwriting advice. This broader workflow report
is distinct from the dashboard's existing race-only reporting contract.


In [ ]:
fairness_report = fairness(test, holdout_scores[preferred_name], thresholds[preferred_name])
display(pd.DataFrame(fairness_report['rows']))
display(pd.DataFrame([
    {'demographic': field, 'metric': metric, **detail}
    for field, measures in fairness_report['group_rate_gaps'].items()
    for metric, detail in measures.items()
]))
print(fairness_report['undefined_rates'])
print(fairness_report['gap_policy'])


## 8. Operational handoff

By default this notebook performs no downloads, artifact writes, fitting, MLflow logging,
or deployment actions. The explicitly enabled historical-ingestion section can download
source archives and write new Parquets into a fresh isolated artifact directory. Model
fitting has a separate opt-in and stays in memory. All outputs are cleared so old results cannot be
mistaken for a fresh execution. Running it may take time for validation permutation importance.

Run operational commands separately from the project root when needed:

| Purpose | Command |
|---|---|
| Validate dashboard artifacts | `python -m scripts.validate_artifacts` |
| Evaluate the existing model and write new Job reports | `python jobs/evaluate_model.py` |
| Track the two existing models in Domino | `python experiments/log_model_comparison.py` |
| Train a new candidate into a unique artifact directory | `python workflows/train_hmda.py` |

See [README](README.md), [source contract](docs/HMDA_SOURCE.md), and
[handoff](docs/HANDOFF.md). New candidates do not automatically replace the dashboard's model.
For a stronger final-validation claim, use a new untouched cohort; renaming or cleaning a
notebook does not change the history of test-set exploration.
